# Köppen-Geiger climate classification for the catchment
This notebook analyses the Köppen-Geiger climate classification of the catchment region defined in `settings.json`. It uses the catchment shapefile to mask the global Köppen-Geiger raster and shows the dominant climate class(es) present in the region.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys
import json
from pathlib import Path
import pandas as pd

from rich import print

In [ ]:
# Import koppen_geiger from scripts — works both on SRC and HPC
try:
    from scripts.koppen_geiger import (
        analyse_koppen_geiger,
        KOPPEN_DESCRIPTION,
    )
except ImportError:
    project_root = Path().resolve().parent
    sys.path.append(str(project_root))
    from scripts.koppen_geiger import (
        analyse_koppen_geiger,
        KOPPEN_DESCRIPTION,
    )

In [ ]:
# Parameters — overridden by papermill when running on HPC
settings_path = "settings.json"

In [ ]:
# Path to the global Köppen-Geiger raster (.tif)
if "ewater" in Path.home():
    # On Spider this is available under /project/ewater/Data/
    # Doing this manually now
    koppen_raster_path = "/home/ewater-mmelotto/koppen_geiger/1991_2020/koppen_geiger_0p00833333.tif"
else:
    koppen_raster_path = "/data/shared/climate-data/koppen_geiger/1991_2020/koppen_geiger_0p00833333.tif"

In [ ]:
# Load settings
with open(settings_path, "r") as f:
    settings = json.load(f)

display(settings)

In [ ]:
# Build shapefile dict for the catchment
shapefiles = {
    settings["caravan_id"]: {
        "path": settings["path_shape"],
        "edgecolor": "red",
        "linewidth": 1.5,
    }
}

In [ ]:
# Run Köppen-Geiger analysis for the catchment
# Saves map, histogram, and table to path_output/koppen_geiger/
save_dir = Path(settings["path_output"]) / "koppen_geiger"

df_percent, top_df = analyse_koppen_geiger(
    path_to_file=koppen_raster_path,
    shapefiles=shapefiles,
    koppen_description=KOPPEN_DESCRIPTION,
    plot_map=True,
    plot_hist=True,
    generate_table=True,
    save_dir=save_dir,
    show_plot=True,
)

In [ ]:
# Extract the catchment column, drop zero-coverage classes, sort by percentage descending
catchment_col = settings["caravan_id"]
catchment_percent = df_percent[catchment_col].copy()
catchment_percent = catchment_percent[catchment_percent > 0].sort_values(ascending=False)

result = pd.DataFrame({
    "Climate class": catchment_percent.index,
    "Description": [KOPPEN_DESCRIPTION.get(c, "") for c in catchment_percent.index],
    "Coverage (%)": catchment_percent.values.round(2),
})

print(f"Köppen-Geiger climate classes for {settings['caravan_id']}:")
display(result)